In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors

# =========================================================
# V5 CORREGIDO
# Idea clave:
# - NO usar lat/lon directos como features
# - Sí usar contexto geoespacial vía estaciones únicas
# - KNN construido sobre estaciones únicas (no sobre todas las filas)
# =========================================================

# =========================
# 1. LOAD TRAINING DATA
# =========================
water_quality = pd.read_csv("../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../data/terraclimate_features_training.csv")

# =========================
# 2. PARSE DATES
# =========================
for df_tmp in [water_quality, landsat, terraclimate]:
    df_tmp["Sample Date"] = pd.to_datetime(df_tmp["Sample Date"], dayfirst=True)

# =========================
# 3. TEMPORAL FEATURES
# =========================
water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear

# =========================
# 4. MERGE TRAINING DATA
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 5. FEATURE ENGINEERING
# =========================
df["nir_swir16_ratio"] = df["nir"] / df["swir16"]
df["nir_swir22_ratio"] = df["nir"] / df["swir22"]
df["green_nir_ratio"] = df["green"] / df["nir"]

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / df["swir22"]

df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]
df["ndmi_day"] = df["NDMI"] * df["dayofyear"]

df["pet_day"] = df["pet"] * df["dayofyear"]
df["pet_month"] = df["pet"] * df["month"]
df["nir_day"] = df["nir"] * df["dayofyear"]
df["swir16_day"] = df["swir16"] * df["dayofyear"]
df["ndmi_month"] = df["NDMI"] * df["month"]

# =========================
# 6. HANDLE MISSING VALUES USING TRAIN MEDIANS
# =========================
train_medians = df.median(numeric_only=True)
df = df.fillna(train_medians)

# =========================
# 7. TARGETS
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

# =========================================================
# 8. BUILD STATION-LEVEL TABLE (UNIQUE LAT/LON ONLY)
#    This is the correction vs previous v5
# =========================================================
station_targets = (
    water_quality.groupby(["Latitude", "Longitude"], as_index=False)[targets]
    .mean()
    .rename(columns={
        "Total Alkalinity": "station_alk_mean",
        "Electrical Conductance": "station_ec_mean",
        "Dissolved Reactive Phosphorus": "station_drp_mean"
    })
)

station_coords = station_targets[["Latitude", "Longitude"]].values
station_target_values = station_targets[
    ["station_alk_mean", "station_ec_mean", "station_drp_mean"]
].values

k = 3
eps = 1e-6

# =========================================================
# 9. TRAIN KNN FEATURES USING UNIQUE STATIONS
#    For each row in train, map it to nearest unique stations.
#    If the row belongs to a known station, exclude that exact station.
# =========================================================
nn_station_train = NearestNeighbors(n_neighbors=min(k + 1, len(station_targets)), metric="euclidean")
nn_station_train.fit(station_coords)

row_coords_train = df[["Latitude", "Longitude"]].values
distances_train, indices_train = nn_station_train.kneighbors(row_coords_train)

# Exclude exact same station (distance == 0) when possible
clean_indices_train = []
clean_distances_train = []

for d_row, i_row in zip(distances_train, indices_train):
    mask_nonzero = d_row > 0
    d_filtered = d_row[mask_nonzero]
    i_filtered = i_row[mask_nonzero]

    if len(i_filtered) < k:
        # fallback if not enough non-zero neighbors
        d_filtered = d_row[:k]
        i_filtered = i_row[:k]
    else:
        d_filtered = d_filtered[:k]
        i_filtered = i_filtered[:k]

    clean_distances_train.append(d_filtered)
    clean_indices_train.append(i_filtered)

clean_distances_train = np.array(clean_distances_train)
clean_indices_train = np.array(clean_indices_train)

weights_train = 1.0 / (clean_distances_train + eps)
neighbor_targets_train = station_target_values[clean_indices_train]

df["knn_station_dist_mean"] = clean_distances_train.mean(axis=1)
df["knn_station_dist_min"] = clean_distances_train.min(axis=1)

df["knn_station_alk_mean"] = neighbor_targets_train[:, :, 0].mean(axis=1)
df["knn_station_ec_mean"] = neighbor_targets_train[:, :, 1].mean(axis=1)
df["knn_station_drp_mean"] = neighbor_targets_train[:, :, 2].mean(axis=1)

df["knn_station_alk_wmean"] = np.sum(neighbor_targets_train[:, :, 0] * weights_train, axis=1) / np.sum(weights_train, axis=1)
df["knn_station_ec_wmean"] = np.sum(neighbor_targets_train[:, :, 1] * weights_train, axis=1) / np.sum(weights_train, axis=1)
df["knn_station_drp_wmean"] = np.sum(neighbor_targets_train[:, :, 2] * weights_train, axis=1) / np.sum(weights_train, axis=1)

# =========================================================
# 10. PREPARE TRAIN MATRICES
#     IMPORTANT: no raw Latitude/Longitude in model
# =========================================================
X = df.drop(columns=targets + ["Sample Date", "Latitude", "Longitude"])
y = df[targets]

# =========================================================
# 11. TRAIN MODEL
# =========================================================
rf_final = RandomForestRegressor(
    n_estimators=700,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

# =========================
# 12. LOAD SUBMISSION DATA
# =========================
submission = pd.read_csv("../data/submission_template.csv")
landsat_val = pd.read_csv("../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../data/terraclimate_features_validation.csv")

for df_tmp in [submission, landsat_val, terraclimate_val]:
    df_tmp["Sample Date"] = pd.to_datetime(df_tmp["Sample Date"], dayfirst=True)

# =========================
# 13. TEMPORAL FEATURES
# =========================
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear

# =========================
# 14. MERGE VALIDATION DATA
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 15. SAME FEATURE ENGINEERING
# =========================
df_val["nir_swir16_ratio"] = df_val["nir"] / df_val["swir16"]
df_val["nir_swir22_ratio"] = df_val["nir"] / df_val["swir22"]
df_val["green_nir_ratio"] = df_val["green"] / df_val["nir"]

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / df_val["swir22"]

df_val["nir_pet"] = df_val["nir"] * df_val["pet"]
df_val["swir16_pet"] = df_val["swir16"] * df_val["pet"]
df_val["ndmi_day"] = df_val["NDMI"] * df_val["dayofyear"]

df_val["pet_day"] = df_val["pet"] * df_val["dayofyear"]
df_val["pet_month"] = df_val["pet"] * df_val["month"]
df_val["nir_day"] = df_val["nir"] * df_val["dayofyear"]
df_val["swir16_day"] = df_val["swir16"] * df_val["dayofyear"]
df_val["ndmi_month"] = df_val["NDMI"] * df_val["month"]

# =========================
# 16. HANDLE MISSING VALUES USING TRAIN MEDIANS
# =========================
df_val = df_val.fillna(train_medians)

# =========================================================
# 17. KNN STATION FEATURES FOR VALIDATION
#     Use only unique stations from training
# =========================================================
row_coords_val = df_val[["Latitude", "Longitude"]].values

nn_station_val = NearestNeighbors(n_neighbors=min(k, len(station_targets)), metric="euclidean")
nn_station_val.fit(station_coords)

distances_val, indices_val = nn_station_val.kneighbors(row_coords_val)

weights_val = 1.0 / (distances_val + eps)
neighbor_targets_val = station_target_values[indices_val]

df_val["knn_station_dist_mean"] = distances_val.mean(axis=1)
df_val["knn_station_dist_min"] = distances_val.min(axis=1)

df_val["knn_station_alk_mean"] = neighbor_targets_val[:, :, 0].mean(axis=1)
df_val["knn_station_ec_mean"] = neighbor_targets_val[:, :, 1].mean(axis=1)
df_val["knn_station_drp_mean"] = neighbor_targets_val[:, :, 2].mean(axis=1)

df_val["knn_station_alk_wmean"] = np.sum(neighbor_targets_val[:, :, 0] * weights_val, axis=1) / np.sum(weights_val, axis=1)
df_val["knn_station_ec_wmean"] = np.sum(neighbor_targets_val[:, :, 1] * weights_val, axis=1) / np.sum(weights_val, axis=1)
df_val["knn_station_drp_wmean"] = np.sum(neighbor_targets_val[:, :, 2] * weights_val, axis=1) / np.sum(weights_val, axis=1)

# =========================================================
# 18. PREPARE VALIDATION FEATURES
# =========================================================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

X_val = X_val[X.columns]

# =========================================================
# 19. PREDICT
# =========================================================
predictions = rf_final.predict(X_val)

# =========================================================
# 20. BUILD SUBMISSION
# =========================================================
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

submission_v5 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

# =========================================================
# 21. EXPORT
# =========================================================
submission_v5.to_csv("../submissions/submission_v5_corrected.csv", index=False)

# =========================================================
# 22. QUICK CHECK
# =========================================================
print(submission_v5.shape)
print(submission_v5.head())

(200, 6)
   Longitude   Latitude Sample Date  Total Alkalinity  Electrical Conductance  \
0  27.822778 -32.043333  2014-09-01        146.373394              529.587229   
1  26.077500 -33.329167  2015-09-16        179.784723              546.992170   
2  27.640028 -32.991639  2015-05-07        129.511368              441.757404   
3  24.439167 -34.096389  2012-02-07        108.693276              527.666399   
4  28.581667 -32.000556  2014-10-01        128.441568              544.086983   

   Dissolved Reactive Phosphorus  
0                      32.819880  
1                      32.144842  
2                      29.643419  
3                      21.075915  
4                      27.942205  
